In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing
from mlflow.models import infer_signature

In [2]:
housing = fetch_california_housing()
features = housing["feature_names"]

In [3]:
x = housing["data"]
y = housing["target"]

In [4]:
df = pd.DataFrame(x, columns=features)

In [5]:
df = pd.concat([df, pd.DataFrame(data={"target": y})], axis=1)

In [6]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [7]:
y.shape

(20640,)

In [8]:
from urllib.parse import urlparse

In [9]:
x = df.drop(["target"], axis=1)
y = df["target"]

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [11]:
signature = infer_signature(x_train, y_train)

In [33]:
rf_regressor_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5,],
    "min_samples_leaf": [1, 2],
}


In [34]:
def hyperparameter_tuning(x_train, y_train, param_grid):
    rf = RandomForestRegressor()
    grid = GridSearchCV(estimator = rf, param_grid = param_grid, cv = 2, n_jobs = -1, verbose = 2, 
    scoring="neg_mean_squared_error")
    grid.fit(x_train, y_train)
    return grid

In [40]:
# MLFLOW Experiments

mlflow.set_tracking_uri("http://127.0.0.1:5000")

with mlflow.start_run():

    # hyperparameter tuning
    grid_search = hyperparameter_tuning(x_train, y_train, rf_regressor_param_grid)

    # best model
    best_model = grid_search.best_estimator_

    # evaluate model
    y_pred = best_model.predict(x_test)

    mse = mean_squared_error(y_pred, y_test)

    # Log best parameters and metrics

    mlflow.log_param("best_n_estimator", grid_search.best_params_["n_estimators"])
    mlflow.log_param("max_depth", grid_search.best_params_["max_depth"])
    mlflow.log_param("min_samples_split", grid_search.best_params_["min_samples_split"])
    mlflow.log_param("min_samples_leaf", grid_search.best_params_["min_samples_leaf"])

    mlflow.log_metric("mse", mse)


    # Tracking uri
    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


    if tracking_url_type_store != 'file':
        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name = "Best RF"
        )

    else:
        mlflow.sklearn.log_model(
            best_model,
            "model",
            signature = signature
        )

    print("mse", mse)

Fitting 2 folds for each of 24 candidates, totalling 48 fits


2025/04/28 22:50:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Best RF'.
2025/04/28 22:50:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best RF, version 1


mse 0.26823007901728735
🏃 View run bouncy-cub-767 at: http://127.0.0.1:5000/#/experiments/0/runs/f4ed6e6701e64f868597a6309af63f5e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


Created version '1' of model 'Best RF'.
